# Phase 2: TF-IDF + K-Means topic clustering
This notebook starts from `reviews_with_sentiment.csv`, generated at the end of Phase 1.

In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path("/content/voxforge-ai-review-analytics")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
%cd $PROJECT_ROOT

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from src.config import *
from src.data.load import load_csv
from src.clustering.tfidf_kmeans import (evaluate_cluster_counts, fit_topic_model, top_terms, representative_reviews, add_topic_labels, topic_sentiment_summary, create_svd_coordinates, save_topic_model)
create_project_directories()

In [ ]:
reviews = load_csv(ENRICHED_REVIEWS_PATH)
TEXT_COLUMN = "classical_text"
model_data = reviews.dropna(subset=[TEXT_COLUMN]).copy()
model_data = model_data[model_data[TEXT_COLUMN].astype(str).str.strip().str.len() > 0].reset_index(drop=True)
print(f"Rows available for clustering: {len(model_data):,}")

## 1. Compare a small range of cluster counts

In [ ]:
cluster_scores = evaluate_cluster_counts(model_data[TEXT_COLUMN], cluster_counts=range(2, 11))
cluster_scores.to_csv(CLUSTERING_RESULTS_DIR / "cluster_count_evaluation.csv", index=False)
display(cluster_scores)
cluster_scores.plot(x="n_clusters", y=["silhouette_score"], marker="o")
plt.title("K-Means silhouette score")
plt.tight_layout(); plt.show()

## 2. Fit the selected cluster count
Choose a value using silhouette score plus manual interpretability. Six is only a starting point.

In [ ]:
N_CLUSTERS = 6
run = fit_topic_model(model_data[TEXT_COLUMN], n_clusters=N_CLUSTERS)
terms = top_terms(run, top_n=12)
representatives = representative_reviews(model_data[TEXT_COLUMN], run, reviews_per_cluster=5)
terms.to_csv(CLUSTERING_RESULTS_DIR / "top_terms.csv", index=False)
representatives.to_csv(CLUSTERING_RESULTS_DIR / "representative_reviews.csv", index=False)
display(terms.groupby("cluster_id")["term"].apply(list).to_frame())
display(representatives)

## 3. Add human-readable topic names
Inspect the terms and representative reviews, then replace these placeholders.

In [ ]:
TOPIC_NAMES = {cluster_id: f"topic_{cluster_id}" for cluster_id in range(N_CLUSTERS)}
clustered = add_topic_labels(model_data, run, TOPIC_NAMES)
clustered.to_csv(CLUSTERED_REVIEWS_PATH, index=False)
summary = topic_sentiment_summary(clustered)
summary.to_csv(CLUSTERING_RESULTS_DIR / "topic_sentiment_summary.csv", index=False)
display(summary)

## 4. Two-dimensional visualization

In [ ]:
coordinates = create_svd_coordinates(run)
coordinates.to_csv(CLUSTERING_RESULTS_DIR / "svd_coordinates.csv", index=False)
plt.figure(figsize=(9, 6))
for cluster_id, group in coordinates.groupby("cluster_id"):
    plt.scatter(group["component_1"], group["component_2"], s=8, alpha=0.5, label=f"Cluster {cluster_id}")
plt.title("Review topics projected with Truncated SVD")
plt.xlabel("Component 1"); plt.ylabel("Component 2"); plt.legend(); plt.tight_layout()
plt.savefig(CLUSTERING_RESULTS_DIR / "topic_clusters_svd.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
model_path = save_topic_model(run, CLUSTERING_MODELS_DIR / "tfidf_kmeans.joblib")
print("Saved clustered reviews:", CLUSTERED_REVIEWS_PATH)
print("Saved topic model:", model_path)